In [1]:
import requests
import calendar
from datetime import datetime
from time import sleep
import pandas as pd

In [25]:
endpoint_url = "https://query.wikidata.org/sparql"
headers = {
    "Accept": "application/sparql-results+json"
}

query = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?event ?eventLabel ?date WHERE {
  SERVICE <https://query.wikidata.org/sparql> {
    ?event wdt:P585 ?date .
    
    FILTER(?date >= "1933-01-02T00:00:00Z"^^xsd:dateTime &&
           ?date <= "1933-12-31T23:59:59Z"^^xsd:dateTime)
           
    OPTIONAL {
      ?event rdfs:label ?eventLabel .
      FILTER(LANG(?eventLabel) = "de" || LANG(?eventLabel) = "en")
    }
  }
}
ORDER BY ?date
LIMIT 1000
"""

query = """
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?event ?eventLabel ?date WHERE {
  SERVICE <https://query.wikidata.org/sparql> {
    ?event wdt:P585 ?date .
    
    FILTER(?date >= "1938-01-02T00:00:00Z"^^xsd:dateTime &&
           ?date <= "1938-12-31T23:59:59Z"^^xsd:dateTime)

    # Ensure it's a historical event or subclass
    ?event wdt:P31/wdt:P279* wd:Q1656682 .
    
    # Filter for events in Germany and the German Reich
    FILTER(?country IN (wd:Q7318, wd:Q28108, wd:Q16957, wd:Q1198, wd:Q183, wd:Q1206012)) .
    ?event wdt:P17 ?country .


    OPTIONAL {
      ?event rdfs:label ?eventLabel .
      FILTER(LANG(?eventLabel) = "de" || LANG(?eventLabel) = "en")
    }
  }
}
ORDER BY ?date
LIMIT 100
"""

In [26]:
response = requests.get(endpoint_url, params={"query": query}, headers=headers)
if response.status_code != 200:
    print(f"Error {response.status_code}: {response.text}")
else:
    data = response.json()

    for item in data["results"]["bindings"]:
        print(item["event"]["value"], item.get("eventLabel", {}).get("value", ""), item["date"]["value"])


http://www.wikidata.org/entity/Q28154369 1937 Tschammerpokal Final 1938-01-09T00:00:00Z
http://www.wikidata.org/entity/Q670166 Eiskunstlauf-Weltmeisterschaft 1938 1938-02-01T00:00:00Z
http://www.wikidata.org/entity/Q670166 1938 World Figure Skating Championships 1938-02-01T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 Reichstagswahl 1938 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q447420 1938 German parliamentary election and referendum 1938-04-10T00:00:00Z
http://www.wikidata.org/entity/Q1532978 Septemberverschwörung 1938-09-01T00:00:00Z
http://www.wikidata.org/entity/Q1532978 Oster Conspiracy 1938-0

In [ ]:
endpoint_url = "https://query.wikidata.org/sparql"
headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "EventFetcher/1.0 (contact: you@example.com)"  # bitte mit eigener Kontaktinfo ersetzen
}

months = [
    "1932-04", "1932-07", "1934-08", "1936-03", "1938-05", "1939-07",
    "1939-07", "1943-05"
]

months = ['1932-04',
 '1932-09',
 '1934-08',
 '1936-03',
 '1940-06',
 '1942-05',
 '1943-01',
 '1943-05',
 '1933-10',
 '1934-01',
 '1934-02',
 '1938-04',
 '1938-09',
 '1939-04',
 '1939-09',
 '1940-05',
 '1932-06',
 '1932-07',
 '1932-10',
 '1933-02']

def month_bounds(yyyymm: str):
    y = int(yyyymm[:4])
    m = int(yyyymm[5:7])
    last_day = calendar.monthrange(y, m)[1]
    start = f"{y}-{m:02d}-01T00:00:00Z"
    end   = f"{y}-{m:02d}-{last_day:02d}T23:59:59Z"
    return start, end

def build_query(start_iso: str, end_iso: str) -> str:
    return f"""
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT ?event ?eventLabel ?date WHERE {{
  ?event wdt:P585 ?date .

  FILTER(?date >= "{start_iso}"^^xsd:dateTime &&
         ?date <= "{end_iso}"^^xsd:dateTime)

  # Instanz von (oder Unterklasse von) historischem Ereignis
  ?event wdt:P31/wdt:P279* wd:Q1656682 .

  # Länderfilter: Deutschland und Deutsches Reich (inkl. historischen IDs)
  ?event wdt:P17 ?country .
  FILTER(?country IN (wd:Q7318, wd:Q28108, wd:Q16957, wd:Q1198, wd:Q183, wd:Q1206012))

  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "de,en".
  }}
}}
ORDER BY ?date
"""

def fetch_month_events(yyyymm: str):
    start_iso, end_iso = month_bounds(yyyymm)
    query = build_query(start_iso, end_iso)
    resp = requests.get(endpoint_url, params={"query": query}, headers=headers, timeout=600)
    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")

    data = resp.json()
    events = []
    for b in data.get("results", {}).get("bindings", []):
        events.append({
            "event": b["event"]["value"],
            "label": b.get("eventLabel", {}).get("value", ""),
            "date":  b["date"]["value"]
        })
    return events

# ---- Hauptlauf: alle Monate sammeln ----
all_events = []
for m in months:
    try:
        month_events = fetch_month_events(m)
        all_events.extend(month_events)
        # höfliche kleine Pause, um Rate Limits zu respektieren
        sleep(0.2)
    except Exception as e:
        print(f"Fehler bei {m}: {e}")

# Optional: global nach Datum sortieren
all_events.sort(key=lambda x: x["date"])

events_df = pd.DataFrame(all_events)

# CSV speichern
events_df.to_csv("wikidata_events.csv", index=False, encoding="utf-8", sep=";")

# Ausgabe / Nutzung
print(f"Gesamt: {len(all_events)} Events")
for ev in all_events[:20]:  # Beispiel: die ersten 20 anzeigen
    print(ev["date"], "-", ev["label"], "-", ev["event"])


Gesamt: 32 Events
1932-07-17T00:00:00Z - Großer Preis von Deutschland 1932 - http://www.wikidata.org/entity/Q2021949
1932-07-17T00:00:00Z - Großer Preis von Deutschland 1932 - http://www.wikidata.org/entity/Q2021949
1933-02-20T00:00:00Z - Geheimtreffen vom 20. Februar 1933 - http://www.wikidata.org/entity/Q1498481
1934-01-01T00:00:00Z - Deutsche Dreiband-Meisterschaft 1934 - http://www.wikidata.org/entity/Q15109305
1934-01-01T00:00:00Z - Norddeutsche Turnmeisterschaften 1934 - http://www.wikidata.org/entity/Q30043239
1934-01-01T00:00:00Z - Bausiedelschau 1934 Frankfurt (Oder) - http://www.wikidata.org/entity/Q81823479
1934-01-01T00:00:00Z - Deutsches Meisterschaftsrudern 1934 - http://www.wikidata.org/entity/Q104600652
1934-01-01T00:00:00Z - Deutsche Fechtmeisterschaften 1934 - http://www.wikidata.org/entity/Q999355
1934-01-01T00:00:00Z - Wasserball-Europameisterschaft 1934 - http://www.wikidata.org/entity/Q774928
1934-01-01T00:00:00Z - UCI-Bahn-Weltmeisterschaften 1934 - http://www.wi

In [3]:
events_df = pd.read_csv("wikidata_events.csv", encoding="utf-8", sep=";")
events_df = events_df.drop_duplicates(subset=['event'])
events_df["date"] = pd.to_datetime(events_df["date"])
events_df = events_df[events_df["date"].dt.strftime("%m-%d") != "01-01"]
events_df

,event,label,date
0,http://www.wikidata.org/entity/Q706684,Reichspräsidentenwahl 1932,1932-04-10 00:00:00+00:00
1,http://www.wikidata.org/entity/Q2021949,Großer Preis von Deutschland 1932,1932-07-17 00:00:00+00:00
3,http://www.wikidata.org/entity/Q1498481,Geheimtreffen vom 20. Februar 1933,1933-02-20 00:00:00+00:00
19,http://www.wikidata.org/entity/Q638482,UCI-Straßen-Weltmeisterschaften 1934,1934-08-18 00:00:00+00:00
20,http://www.wikidata.org/entity/Q4305909,Volksabstimmung über das Staatsoberhaupt des D...,1934-08-19 00:00:00+00:00
21,http://www.wikidata.org/entity/Q389024,Reichstagswahl im Deutschen Reich 1936,1936-03-29 00:00:00+00:00
23,http://www.wikidata.org/entity/Q1532978,Septemberverschwörung,1938-09-01 00:00:00+00:00
24,http://www.wikidata.org/entity/Q2378829,Deutsch-sowjetische Siegesparade in Brest-Litowsk,1939-09-22 00:00:00+00:00
52,http://www.wikidata.org/entity/Q447420,Reichstagswahl 1938,1938-04-10 00:00:00+00:00


In [4]:
def join_unique(series, sep):
    seen = set()
    out = []
    for x in series.astype(str):
        x = x.strip()
        if not x or x in seen:
            continue
        seen.add(x)
        out.append(x)
    return sep.join(out)

# 1) Variante: Events und Labels als getrennte, zusammengefasste Spalten
agg_df = (
    events_df
      .groupby('date', sort=False)
      .agg(
          events=('event', lambda s: join_unique(s, " | ")),   # z.B. „Event A | Event B | …”
          labels=('label', lambda s: join_unique(s.fillna(""), "\n\n"))  # Absätze zwischen Labels
      )
      .reset_index()
)

# 2) (Optional) Eine einzige Spalte, die Events + Labels pro Datum zusammenführt
#    Events in einer Zeile, dann eine Leerzeile, dann alle Labels mit Absätzen.
agg_df['event_label'] = agg_df.apply(
    lambda r: f"{r['events']}\n\n{r['labels']}" if r['labels'] else r['events'],
    axis=1
)

agg_df

,date,events,labels,event_label
0,1932-04-10 00:00:00+00:00,http://www.wikidata.org/entity/Q706684,Reichspräsidentenwahl 1932,http://www.wikidata.org/entity/Q706684\n\nReic...
1,1932-07-17 00:00:00+00:00,http://www.wikidata.org/entity/Q2021949,Großer Preis von Deutschland 1932,http://www.wikidata.org/entity/Q2021949\n\nGro...
2,1933-02-20 00:00:00+00:00,http://www.wikidata.org/entity/Q1498481,Geheimtreffen vom 20. Februar 1933,http://www.wikidata.org/entity/Q1498481\n\nGeh...
3,1934-08-18 00:00:00+00:00,http://www.wikidata.org/entity/Q638482,UCI-Straßen-Weltmeisterschaften 1934,http://www.wikidata.org/entity/Q638482\n\nUCI-...
4,1934-08-19 00:00:00+00:00,http://www.wikidata.org/entity/Q4305909,Volksabstimmung über das Staatsoberhaupt des D...,http://www.wikidata.org/entity/Q4305909\n\nVol...
5,1936-03-29 00:00:00+00:00,http://www.wikidata.org/entity/Q389024,Reichstagswahl im Deutschen Reich 1936,http://www.wikidata.org/entity/Q389024\n\nReic...
6,1938-09-01 00:00:00+00:00,http://www.wikidata.org/entity/Q1532978,Septemberverschwörung,http://www.wikidata.org/entity/Q1532978\n\nSep...
7,1939-09-22 00:00:00+00:00,http://www.wikidata.org/entity/Q2378829,Deutsch-sowjetische Siegesparade in Brest-Litowsk,http://www.wikidata.org/entity/Q2378829\n\nDeu...
8,1938-04-10 00:00:00+00:00,http://www.wikidata.org/entity/Q447420,Reichstagswahl 1938,http://www.wikidata.org/entity/Q447420\n\nReic...
